Импортируем библиотеки

In [1]:
import nltk
import random
import numpy as np
import pandas as pd
import pprint, time
from sklearn.model_selection import train_test_split

In [15]:
from tqdm import notebook

Скачиваем данные из Github

In [19]:
!wget https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
!wget https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt

--2024-11-07 21:28:34--  https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7626752 (7.3M) [text/plain]
Saving to: ‘GSD_train.txt.1’

GSD_train.txt.1     100%[===================>]   7.27M   623KB/s    in 13s     

2024-11-07 21:28:47 (584 KB/s) - ‘GSD_train.txt.1’ saved [7626752/7626752]

--2024-11-07 21:28:48--  https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81386 (79K) [t

In [2]:
with open("GSD_train.txt", encoding='utf-8') as f:
  data = f.read()

In [3]:
sent = data.split('\n\n')

Представляем данные в виде списка, который содержит другие списки

In [4]:
tokens = []
for sentence in sent:
    s = []
    for word in sentence.split('\n'):
        if word:
            token = (word.split()[1], word.split()[3])
        s.append(token)
    tokens.append(s)

Подготовка к обучению HMM

In [5]:
train_set,test_set =train_test_split(tokens, train_size=0.80, test_size=0.20, random_state = 101)

Создаем список размеченных обучающих и тестовых данных

In [6]:
train_tagged_words = [ tup for sent in train_set for tup in sent ]
test_tagged_words = [ tup for sent in test_set for tup in sent ]

Рассчитываем вероятности эмиссии
они определяют вероятность увидеть определенную наблюдаемую переменную при заданном значении для скрытых переменных,
т.е. эта функция возвращает то, сколько всего раз тег встречался в выборке и сколько раз определенное слово встречалось с этим тегом

In [7]:
def word_given_tag(word, tag, train_bag=train_tagged_words):
    tag_list = [pair for pair in train_bag if pair[1]==tag] # список всех слов определенного тега
    count_tag = len(tag_list)# общее число появления тегов в обучающей выборке
    w_given_tag_list = [pair[0] for pair in tag_list if pair[0]==word] # cписок, который состоит из всех вхождений данного слова, которые были помечены определенным тегом
    count_w_given_tag = len(w_given_tag_list) # подсчитываем общее количество раз, когда слово встречалось с тегом

    return (count_w_given_tag, count_tag)

Рассчитываем вероятность переходов

In [8]:
def t2_given_t1(t2, t1, train_bag=train_tagged_words): # t1 = t2, это пронумерованный список тегов
    tags = [pair[1] for pair in train_bag] # список всех тегов по порядку (просто из общего спсика размеченных слов мы убираем слова, получаем последовательность тегов)
    count_t1 = len([t for t in tags if t==t1])
    count_t2_t1 = 0
    for index in range(len(tags)-1):
        if tags[index]==t1 and tags[index+1] == t2: # если при теге t1 следующий тег = t2, то осуществляется переход
            count_t2_t1 += 1
    return (count_t2_t1, count_t1) # сколько всего раз встречается данный (второе значение), сколько раз данный тег переходит в другой тег (первое значение)

создаем матрицу тегов t x t, где t - номер тега
матрица(i, j) представляет вероятность перехода P(i-й тег переходит в j-й тег)

In [9]:
tags = list(set([pair[1] for pair in train_tagged_words]))
tags_matrix = np.zeros((len(tags), len(tags)), dtype='float32')
for i, t1 in enumerate(list(tags)):
    for j, t2 in enumerate(list(tags)):
        tags_matrix[i, j] = t2_given_t1(t2, t1)[0]/t2_given_t1(t2, t1)[1]

print(tags_matrix)

[[8.16326495e-03 1.06122447e-02 1.63265306e-03 7.14285731e-01
  3.26530612e-03 0.00000000e+00 1.26530617e-01 4.89795916e-02
  1.95918363e-02 1.63265299e-02 9.79591813e-03 3.26530612e-03
  5.71428565e-03 2.44897953e-03 2.44897953e-03 2.69387756e-02]
 [4.20673098e-03 1.38221150e-02 3.00480775e-03 5.49879789e-01
  1.20192312e-03 4.20673080e-02 9.37500000e-02 3.00480775e-03
  1.50240380e-02 1.79086536e-01 4.20673098e-03 0.00000000e+00
  6.00961561e-04 6.00961549e-03 1.68269239e-02 6.73076957e-02]
 [6.31911540e-03 1.10584516e-02 4.73933667e-03 8.21484998e-02
  6.16113730e-02 0.00000000e+00 1.10584520e-01 2.21169032e-02
  6.60347581e-01 7.89889414e-03 6.31911540e-03 0.00000000e+00
  3.15955770e-03 1.57977885e-03 4.73933667e-03 1.73775665e-02]
 [1.01723652e-02 1.29038338e-02 1.19619481e-02 1.51549399e-01
  1.23857968e-02 1.31863996e-03 1.11330882e-01 6.64029410e-03
  8.39691088e-02 3.34039748e-01 7.56805092e-02 1.88377133e-04
  7.67636811e-03 9.32466798e-03 4.21964787e-02 1.28661588e-01]
 [1.

In [10]:
tags_df = pd.DataFrame(tags_matrix, columns = list(tags), index=list(tags))
display(tags_df)

,DET,NUM,AUX,NOUN,ADV,SYM,ADJ,PART,VERB,PUNCT,PROPN,SCONJ,PRON,X,CCONJ,ADP
DET,0.008163,0.010612,0.001633,0.714286,0.003265,0.000000,0.126531,0.048980,0.019592,0.016327,0.009796,0.003265,0.005714,0.002449,0.002449,0.026939
NUM,0.004207,0.013822,0.003005,0.549880,0.001202,0.042067,0.093750,0.003005,0.015024,0.179087,0.004207,0.000000,0.000601,0.006010,0.016827,0.067308
AUX,0.006319,0.011058,0.004739,0.082148,0.061611,0.000000,0.110585,0.022117,0.660348,0.007899,0.006319,0.000000,0.003160,0.001580,0.004739,0.017378
NOUN,0.010172,0.012904,0.011962,0.151549,0.012386,0.001319,0.111331,0.006640,0.083969,0.334040,0.075681,0.000188,0.007676,0.009325,0.042196,0.128662
ADV,0.013498,0.032621,0.013498,0.061867,0.048931,0.000562,0.128234,0.048931,0.288526,0.116985,0.013498,0.006187,0.036558,0.003375,0.021372,0.165354
SYM,0.006944,0.131944,0.000000,0.263889,0.000000,0.041667,0.076389,0.000000,0.090278,0.180556,0.020833,0.000000,0.006944,0.090278,0.006944,0.083333
ADJ,0.002165,0.001959,0.002990,0.736365,0.002681,0.000206,0.085679,0.002268,0.012682,0.085679,0.019383,0.000103,0.001547,0.000928,0.021961,0.023404
PART,0.019906,0.030445,0.026932,0.135831,0.064403,0.000000,0.106557,0.033958,0.385246,0.031616,0.044496,0.003513,0.010539,0.000000,0.000000,0.106557
VERB,0.028060,0.038358,0.003731,0.234179,0.035522,0.000448,0.149552,0.016866,0.049403,0.079254,0.024776,0.000597,0.026418,0.002687,0.010000,0.300149
PUNCT,0.013449,0.029350,0.005184,0.165803,0.047352,0.001121,0.124265,0.009456,0.099187,0.118310,0.094424,0.027108,0.034604,0.027178,0.051625,0.151513


Алгоритм Витерби

In [16]:
def Viterbi(words, train_bag=train_tagged_words): # первый аргумент - список слов, которым нам нужно присвоить тег, второй - размеченная обучающая выборка
    state = []
    T = list(set([pair[1] for pair in train_bag])) # список всех возможных тегов

    for key, word in enumerate(notebook.tqdm(words)): # пронумеровываем список слов, чтобы можно было смотреть на предыдущее
        p = [] # инициализируем список столбцов вероятности для каждого наблюдения
        for tag in T: # считаем вероятность перехода
            if key == 0: # если предыдущего тега нет, то рассматриваем вероятность появления слова после знака препинания, в начале предложения
                transition_p = tags_df.loc['PUNCT', tag]
            else: # если предыдущий тег есть, то рассматриваем его
                transition_p = tags_df.loc[state[-1], tag]

            # вычисляем вероятности выбросов и состояний
            emission_p = word_given_tag(words[key], tag)[0]/word_given_tag(words[key], tag)[1] # вероятность выбросов ( насколько вероятно, что конкретное слово будет иметь конкретный тег)
            state_probability = emission_p * transition_p  # вероятность состояний (умножаем веротность того, что слово в принципе встречалось с этим тегом на вероятность перехода)
            p.append(state_probability)


        pmax = max(p) # выбираем максимальное значение из полученного списка
        # получаем наиболее вероятную последовательность скрытых состояний
        state_max = T[p.index(pmax)] # записываем его индекс и находим его соответствие в списке тегов
        state.append(state_max)
    return list(zip(words, state))

In [17]:
# протестируем алгоритм Витерби на нескольких примерах предложений тестового набора данных
random.seed(1234)      #определяем случайное значение

# выбираем случайные 10 значений
rndom = [random.randint(1,len(test_set)) for x in range(50)]

# список из 10 предложений, на которых мы тестируем модель
test_run = [test_set[i] for i in rndom]

# список размеченных слов
test_run_base = [tup for sent in test_run for tup in sent]

# список неразмеченных слов
test_tagged_words = [tup[0] for sent in test_run for tup in sent]

Проверяем скорость и точность алгоритма

In [18]:
start = time.time()
tagged_seq = Viterbi(test_tagged_words)
end = time.time() # считаем сколько времени это заняло
difference = end-start

print("Время в секундах: ", difference)

# проходим по парам (разметка_HMM, разметка_эталон) и если совпали, сохраняем в список
# по существу, нам просто нужно получить количество совпадений
check = [i for i, j in zip(tagged_seq, test_run_base) if i == j]

accuracy = len(check)/len(tagged_seq) # считаем точность
print('Точность алгоритма Витерби, %: ',accuracy*100)

  0%|          | 0/886 [00:00<?, ?it/s]

Время в секундах:  151.24469780921936
Точность алгоритма Витерби, %:  68.51015801354401


Ручная проверка 

In [19]:
for our, reference in list(zip(tagged_seq, test_run_base))[:12]:
  print('Наша: {0:20}\tЭталон: {1:20}'.format(str(our), str(reference)))

Наша: ('Двери', 'DET')    	Эталон: ('Двери', 'NOUN')   
Наша: ('из', 'ADP')       	Эталон: ('из', 'ADP')       
Наша: ('вагонов', 'NOUN') 	Эталон: ('вагонов', 'NOUN') 
Наша: ('поезда', 'NOUN')  	Эталон: ('поезда', 'NOUN')  
Наша: ('не', 'PART')      	Эталон: ('не', 'PART')      
Наша: ('открываются', 'DET')	Эталон: ('открываются', 'VERB')
Наша: ('.', 'PUNCT')      	Эталон: ('.', 'PUNCT')      
Наша: ('Назвав', 'DET')   	Эталон: ('Назвав', 'VERB')  
Наша: ('призванную', 'DET')	Эталон: ('призванную', 'VERB')
Наша: ('девушку', 'NOUN') 	Эталон: ('девушку', 'NOUN') 
Наша: ('в', 'ADP')        	Эталон: ('в', 'ADP')        
Наша: ('честь', 'NOUN')   	Эталон: ('честь', 'NOUN')   


Автоматизированная проверка 

In [51]:
# test_run_base =     [tup       for sent in test_run for tup in sent]
with_sent_numbers = [(i, *tup) for i, sent in enumerate(test_run) for tup in sent]
with_sent_numbers

[(0, 'Двери', 'NOUN'),
 (0, 'из', 'ADP'),
 (0, 'вагонов', 'NOUN'),
 (0, 'поезда', 'NOUN'),
 (0, 'не', 'PART'),
 (0, 'открываются', 'VERB'),
 (0, '.', 'PUNCT'),
 (1, 'Назвав', 'VERB'),
 (1, 'призванную', 'VERB'),
 (1, 'девушку', 'NOUN'),
 (1, 'в', 'ADP'),
 (1, 'честь', 'NOUN'),
 (1, 'своего', 'DET'),
 (1, 'кота', 'NOUN'),
 (1, '``', 'PUNCT'),
 (1, 'Танарот', 'PROPN'),
 (1, '&#39;&#39;', 'PUNCT'),
 (1, ',', 'PUNCT'),
 (1, 'он', 'PRON'),
 (1, 'тем', 'PRON'),
 (1, 'самым', 'ADJ'),
 (1, 'заключает', 'VERB'),
 (1, 'с', 'ADP'),
 (1, 'ней', 'PRON'),
 (1, 'контракт', 'NOUN'),
 (1, 'хозяина', 'NOUN'),
 (1, 'и', 'CCONJ'),
 (1, 'слуги', 'NOUN'),
 (1, '.', 'PUNCT'),
 (2, 'Председатель', 'NOUN'),
 (2, 'Социал-демократической', 'ADJ'),
 (2, 'партии', 'NOUN'),
 (2, 'Боснии', 'PROPN'),
 (2, 'и', 'CCONJ'),
 (2, 'Герцеговины', 'PROPN'),
 (2, '.', 'PUNCT'),
 (3, 'Телефонный', 'ADJ'),
 (3, 'код', 'NOUN'),
 (3, '--', 'PUNCT'),
 (3, '4739', 'NUM'),
 (3, '.', 'PUNCT'),
 (4, 'В', 'ADP'),
 (4, '1942', 'ADJ'),
 

In [38]:
kwiks = []
for sent_i, tk, pos in with_sent_numbers:
    current_sentence = [tk for sent_j, tk, pos in with_sent_numbers if sent_j==sent_i]
    tk_index_in_sent = current_sentence.index(tk)
    kwik = current_sentence[max(0, tk_index_in_sent-3):min(len(sent)+1, tk_index_in_sent+4)]
    kwiks.append((tk, pos, ' '.join(current_sentence), ' '.join(kwik)))

In [ ]:
mismatches = []
for our, reference in zip (tagged_seq, kwiks):
    # print(our, reference)
    if our[1] != reference[1]:
        mismatches.append((our[0], our[1], *reference[1:]))  
mismatches_df = pd.DataFrame(mismatches, columns=['Слово', 'Наша разметка', 'Эталонная разметка', 'SENT_CONTEXT', '3_WIK'])
mismatches_df.index = range(1, len(mismatches_df) + 1)
mismatches_df.to_excel('Mismatches.xlsx', index=False)
display(mismatches_df)

,Слово,Наша разметка,Эталонная разметка,SENT_CONTEXT,3_WIK
1,Двери,DET,NOUN,Двери из вагонов поезда не открываются .,Двери из вагонов поезда
2,открываются,DET,VERB,Двери из вагонов поезда не открываются .,вагонов поезда не открываются .
3,Назвав,DET,VERB,Назвав призванную девушку в честь своего кота ...,Назвав призванную девушку в
4,призванную,DET,VERB,Назвав призванную девушку в честь своего кота ...,Назвав призванную девушку в честь
5,кота,DET,NOUN,Назвав призванную девушку в честь своего кота ...,в честь своего кота `` Танарот &#39;&#39;
...,...,...,...,...,...
275,и,PART,CCONJ,Окисляется на воздухе или по действием иода до...,"иода до азотистой и азотной кислот ,"
276,азотной,DET,ADJ,Окисляется на воздухе или по действием иода до...,"до азотистой и азотной кислот , под"
277,кислот,DET,NOUN,Окисляется на воздухе или по действием иода до...,"азотистой и азотной кислот , под действием"
278,перманганата,DET,NOUN,Окисляется на воздухе или по действием иода до...,", под действием перманганата -- до азотной"


In [52]:
mismatches_df[['Наша разметка', 'Эталонная разметка']].value_counts()

Наша разметка  Эталонная разметка
DET            NOUN                  90
               PROPN                 48
               ADJ                   47
               VERB                  42
PART           CCONJ                 15
DET            NUM                   14
               X                      7
NUM            ADJ                    3
DET            ADV                    2
PRON           DET                    1
DET            SYM                    1
ADJ            NUM                    1
ADP            ADV                    1
DET            ADP                    1
CCONJ          NOUN                   1
AUX            VERB                   1
ADV            ADP                    1
               ADJ                    1
ADP            PROPN                  1
VERB           ADP                    1
Name: count, dtype: int64

In [45]:
mismatches = []
for our, reference in zip (tagged_seq, test_run_base):
    if our != reference:  
        mismatches.append((our[0], our[1], reference[1]))  
mismatches_df = pd.DataFrame(mismatches, columns=['Слово', 'Наша разметка', 'Эталонная разметка'])
mismatches_df.index = range(1, len(mismatches_df) + 1)
mismatches_df.to_excel('Mismatches.xlsx', index=False)
display (mismatches_df)

,Слово,Наша разметка,Эталонная разметка
1,Двери,DET,NOUN
2,открываются,DET,VERB
3,Назвав,DET,VERB
4,призванную,DET,VERB
5,кота,DET,NOUN
...,...,...,...
275,и,PART,CCONJ
276,азотной,DET,ADJ
277,кислот,DET,NOUN
278,перманганата,DET,NOUN
